# Module 4: Function Approximation
## Notebook 2: Semi-Gradient TD(0) with Linear Function Approximation

**Learning Objectives**

By completing this notebook, you will:
1. Derive and implement the semi-gradient TD(0) update for linear function approximation
2. Implement polynomial and Fourier basis features for continuous state spaces
3. Apply linear TD to Mountain Car and visualize the learned value function surface
4. Compare tile coding, polynomial, and Fourier bases on convergence speed and approximation quality

**Prerequisites**
- TD(0) prediction (Module 3, Notebook 1)
- Tile coding (Module 4, Notebook 1)
- Basic linear algebra (dot products, gradients)

**Estimated Time: 14 minutes**

---

### Semi-Gradient TD(0): The Math

With linear function approximation, we parameterize the value function as:

$$\hat{V}(s; \mathbf{w}) = \mathbf{w}^\top \phi(s)$$

where $\phi(s)$ is the feature vector and $\mathbf{w}$ is the weight vector we learn.

The semi-gradient TD(0) update is:

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \underbrace{\left[ R_{t+1} + \gamma \hat{V}(S_{t+1}; \mathbf{w}) - \hat{V}(S_t; \mathbf{w}) \right]}_{\delta_t \text{ (TD error)}} \nabla_{\mathbf{w}} \hat{V}(S_t; \mathbf{w})$$

For linear approximation, $\nabla_{\mathbf{w}} \hat{V}(s; \mathbf{w}) = \phi(s)$, so the update simplifies to:

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \delta_t \phi(S_t)$$

It's called "semi-gradient" because the gradient of the target ($\gamma \hat{V}(S_{t+1}; \mathbf{w})$) with respect to $\mathbf{w}$ is **not** taken — we treat the bootstrap target as a fixed constant.

In [ ]:
# Purpose: Import dependencies and set seeds
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import gymnasium as gym

np.random.seed(42)

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("Setup complete.")

# Mountain Car constants
LOW  = np.array([-1.2,  -0.07])
HIGH = np.array([ 0.6,   0.07])
print(f"Mountain Car state space: position [{LOW[0]}, {HIGH[0]}], velocity [{LOW[1]:.3f}, {HIGH[1]:.3f}]")

## 1. Feature Representations

We implement three feature types and compare them as bases for linear value approximation.

### 1a. Tile Coding Features (from Notebook 1)

In [ ]:
# Purpose: Reproduce TileCoder from Notebook 1 (self-contained notebook)
# Key Concept: Tile coding is the standard basis for linear RL on continuous spaces

class TileCoder:
    """Multi-dimensional tile coding. See Notebook 1 for detailed documentation."""

    def __init__(self, low, high, n_tiles, n_tilings):
        self.low       = np.array(low, dtype=float)
        self.high      = np.array(high, dtype=float)
        self.n_tiles   = n_tiles
        self.n_tilings = n_tilings
        self.n_dims    = len(low)
        self.tile_width = (self.high - self.low) / n_tiles
        self.n_features = n_tilings * (n_tiles ** self.n_dims)

    def encode(self, state):
        state = np.clip(state, self.low, self.high - 1e-8)
        phi   = np.zeros(self.n_features)
        tiles_per_tiling = self.n_tiles ** self.n_dims

        for t_idx in range(self.n_tilings):
            offset     = (t_idx / self.n_tilings) * self.tile_width
            shifted    = state - self.low + offset
            tile_c     = np.clip(np.floor(shifted / self.tile_width).astype(int),
                                 0, self.n_tiles - 1)
            flat_idx   = 0
            for coord in tile_c:
                flat_idx = flat_idx * self.n_tiles + coord
            phi[t_idx * tiles_per_tiling + flat_idx] = 1.0

        return phi


tc = TileCoder(LOW, HIGH, n_tiles=8, n_tilings=8)
print(f"Tile coding: {tc.n_features} features")

### 1b. Polynomial Basis Features

Polynomial features of order $n$ for a 2D state $(x_1, x_2)$ include all products $x_1^i x_2^j$ where $i + j \le n$. First normalize the state to $[-1, 1]$ before computing products.

In [ ]:
# Purpose: Implement polynomial basis features
# Key Concept: Polynomial features can capture smooth value functions exactly

class PolynomialBasis:
    """
    Polynomial basis features of order <= max_order for a 2D state.
    State is normalized to [-1, 1] before computing features.
    Includes a bias term (constant 1.0).
    """

    def __init__(self, low, high, max_order=5):
        self.low       = np.array(low, dtype=float)
        self.high      = np.array(high, dtype=float)
        self.max_order = max_order

        # Pre-compute the exponent pairs (i, j) with i + j <= max_order
        self.exponents = [(i, j)
                          for i in range(max_order + 1)
                          for j in range(max_order + 1 - i)]
        self.n_features = len(self.exponents)

    def normalize(self, state):
        """Map state from [low, high] to [-1, 1]."""
        return 2.0 * (state - self.low) / (self.high - self.low) - 1.0

    def encode(self, state):
        """
        Compute polynomial features for a 2D state.

        Returns
        -------
        phi : np.ndarray, shape (n_features,)
        """
        x = self.normalize(state)
        phi = np.array([x[0]**i * x[1]**j for (i, j) in self.exponents])
        return phi


poly = PolynomialBasis(LOW, HIGH, max_order=5)
print(f"Polynomial basis (order<=5): {poly.n_features} features")
print(f"Exponent pairs: {poly.exponents[:10]}...")

# Test
s = np.array([-0.6, 0.0])
phi_poly = poly.encode(s)
print(f"Sample state {s}: phi has {len(phi_poly)} features, first 5: {phi_poly[:5].round(4)}")

### 1c. Fourier Basis Features

Fourier basis features use cosine functions as the basis:

$$\phi_i(s) = \cos\left( \pi \mathbf{c}_i^\top s' \right)$$

where $s'$ is the normalized state in $[0, 1]^d$ and $\mathbf{c}_i$ is a frequency vector with integer components in $[0, \text{order}]$. Fourier bases are particularly well-suited to smooth functions and can be analyzed by frequency.

In [ ]:
# Purpose: Implement Fourier basis features
# Key Concept: Fourier features decompose the value function by frequency components

class FourierBasis:
    """
    Cosine Fourier basis for 2D continuous states.
    phi_i(s) = cos(pi * c_i^T * s_normalized)
    where s_normalized in [0, 1]^d.
    """

    def __init__(self, low, high, order=5):
        self.low   = np.array(low, dtype=float)
        self.high  = np.array(high, dtype=float)
        self.order = order

        # All frequency vectors c = (c1, c2) with ci in [0, order]
        self.c_vectors = np.array(
            [(c1, c2)
             for c1 in range(order + 1)
             for c2 in range(order + 1)],
            dtype=float
        )  # shape (n_features, 2)
        self.n_features = len(self.c_vectors)

    def normalize(self, state):
        """Map state to [0, 1]."""
        return (state - self.low) / (self.high - self.low)

    def encode(self, state):
        """
        Compute Fourier basis features.

        Returns
        -------
        phi : np.ndarray, shape (n_features,)
        """
        s_norm = self.normalize(state)   # in [0, 1]^2
        # cos(pi * c^T * s) for each frequency vector c
        phi = np.cos(np.pi * self.c_vectors @ s_norm)
        return phi


fourier = FourierBasis(LOW, HIGH, order=5)
print(f"Fourier basis (order=5): {fourier.n_features} features")

s = np.array([-0.6, 0.0])
phi_f = fourier.encode(s)
print(f"Sample state {s}: phi has {len(phi_f)} features, first 5: {phi_f[:5].round(4)}")

## 2. Semi-Gradient TD(0) Implementation

The weight update is the same regardless of which basis we use — only the feature extraction changes. This is the power of the linear framework: swap out `encoder.encode(state)` for any function and the algorithm still works.

In [ ]:
# Purpose: Implement semi-gradient TD(0) prediction with a given encoder
# Key Concept: The TD update w += alpha * delta * phi(s) is identical for all linear bases

def semi_gradient_td0(encoder, n_episodes, alpha, gamma=1.0, seed=42):
    """
    Semi-gradient TD(0) prediction on MountainCar-v0 using a random policy.

    Parameters
    ----------
    encoder    : object with .encode(state) -> np.ndarray and .n_features attribute
    n_episodes : int
    alpha      : float — learning rate
    gamma      : float — discount factor

    Returns
    -------
    w           : np.ndarray, shape (n_features,) — learned weight vector
    ep_lengths  : list — episode length (steps) per episode
    """
    env = gym.make('MountainCar-v0')

    # Step 1: Initialize weights to zero
    w = np.zeros(encoder.n_features)
    ep_lengths = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        done   = False
        steps  = 0

        while not done:
            # Step 2: Encode current state
            phi_s = encoder.encode(obs)

            # Step 3: Random policy (for prediction)
            action = env.action_space.sample()

            # Step 4: Step environment
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            steps += 1

            # Step 5: Compute TD error
            V_s      = np.dot(w, phi_s)
            if done:
                V_next = 0.0   # terminal state has value 0
            else:
                phi_ns = encoder.encode(next_obs)
                V_next = np.dot(w, phi_ns)

            td_error = reward + gamma * V_next - V_s

            # Step 6: Semi-gradient update: w += alpha * delta * phi(s)
            w += alpha * td_error * phi_s

            obs = next_obs

        ep_lengths.append(steps)

    env.close()
    return w, ep_lengths


print("Training with tile coding...")
w_tc, lengths_tc = semi_gradient_td0(tc, n_episodes=200, alpha=0.1 / tc.n_tilings)
print(f"Tile coding: mean episode length = {np.mean(lengths_tc[-20:]):.0f} steps")

print("Training with polynomial basis...")
w_poly, lengths_poly = semi_gradient_td0(poly, n_episodes=200, alpha=1e-4)
print(f"Polynomial: mean episode length = {np.mean(lengths_poly[-20:]):.0f} steps")

print("Training with Fourier basis...")
w_fourier, lengths_fourier = semi_gradient_td0(fourier, n_episodes=200, alpha=1e-4)
print(f"Fourier:    mean episode length = {np.mean(lengths_fourier[-20:]):.0f} steps")

## 3. Visualizing Learned Value Functions

We evaluate the learned weights across a grid of (position, velocity) points to produce a 2D surface plot of $\hat{V}(s; \mathbf{w})$.

In [ ]:
# Purpose: Plot the learned value function surface V(position, velocity)
# Key Concept: The surface should show higher values near the goal (right, positive velocity)

def compute_value_surface(encoder, w, n_grid=40):
    """
    Evaluate V(s; w) = w^T phi(s) on a grid of states.

    Returns
    -------
    pos_grid : np.ndarray, shape (n_grid,)
    vel_grid : np.ndarray, shape (n_grid,)
    V_grid   : np.ndarray, shape (n_grid, n_grid)
    """
    pos_grid = np.linspace(LOW[0], HIGH[0], n_grid)
    vel_grid = np.linspace(LOW[1], HIGH[1], n_grid)
    V_grid   = np.zeros((n_grid, n_grid))

    for i, pos in enumerate(pos_grid):
        for j, vel in enumerate(vel_grid):
            s = np.array([pos, vel])
            V_grid[j, i] = np.dot(w, encoder.encode(s))

    return pos_grid, vel_grid, V_grid


fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw={'projection': '3d'})

configs = [
    (tc,      w_tc,      'Tile Coding'),
    (poly,    w_poly,    'Polynomial Basis (order=5)'),
    (fourier, w_fourier, 'Fourier Basis (order=5)'),
]

for ax, (encoder, w, title) in zip(axes, configs):
    pos_g, vel_g, V_g = compute_value_surface(encoder, w, n_grid=35)
    POS, VEL = np.meshgrid(pos_g, vel_g)

    ax.plot_surface(POS, VEL, V_g, cmap='viridis', alpha=0.85, linewidth=0)
    ax.set_xlabel('Position', fontsize=9)
    ax.set_ylabel('Velocity', fontsize=9)
    ax.set_zlabel('V(s)', fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.view_init(elev=30, azim=-60)

plt.suptitle('Learned Value Functions: V(position, velocity) after 200 Episodes', fontsize=13)
plt.tight_layout()
plt.savefig('../resources/linear_td_value_surfaces.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Convergence Comparison

In [ ]:
# Purpose: Compare learning speed by plotting episode length curves
# Key Concept: Shorter episodes under random policy = faster coverage of state space

def smooth(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')


fig, ax = plt.subplots(figsize=(12, 5))

for lengths, label, color in [
    (lengths_tc,      'Tile Coding',                'steelblue'),
    (lengths_poly,    'Polynomial Basis (order=5)',  'tomato'),
    (lengths_fourier, 'Fourier Basis (order=5)',     'forestgreen'),
]:
    ax.plot(smooth(lengths), color=color, linewidth=2, label=label)

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Length (steps, random policy)', fontsize=12)
ax.set_title('Feature Basis Comparison: Episode Lengths During Prediction\n'
             '(Random policy — shorter = more state space explored efficiently)', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../resources/linear_td_convergence.png', dpi=120, bbox_inches='tight')
plt.show()

### Exercise 4.2.1: Fourier Basis Order vs. Approximation Quality

**Task:** Train semi-gradient TD(0) with Fourier basis for orders [1, 3, 5, 7]. After training, compute the variance of the learned value surface (across all grid points) for each order. Higher order should capture more detail, resulting in higher variance in the value surface. Plot order vs. surface variance.

**Expected Output:** A line plot showing surface variance increasing with Fourier order.

<details>
<summary>Hint 1</summary>
Use FourierBasis(LOW, HIGH, order=k) for each k, train with semi_gradient_td0, then compute_value_surface and np.var(V_grid) to get the variance.
</details>

<details>
<summary>Hint 2</summary>
Use alpha=1e-4 for all orders. Higher orders have more features but the same alpha, so they may need more episodes to converge. Use n_episodes=150 and focus on the trend.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

orders = [1, 3, 5, 7]
surface_variances = []
n_features_list   = []

for order in orders:
    fb = FourierBasis(LOW, HIGH, order=order)
    w_f, _ = semi_gradient_td0(fb, n_episodes=150, alpha=1e-4, seed=0)
    _, _, V_g = compute_value_surface(fb, w_f, n_grid=30)
    surface_variances.append(np.var(V_g))
    n_features_list.append(fb.n_features)
    print(f"Order {order}: {fb.n_features} features, V surface variance = {surface_variances[-1]:.4f}")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(orders, surface_variances, 'o-', color='steelblue',
         linewidth=2, markersize=9, label='Surface variance')
ax2.bar(orders, n_features_list, width=0.5, alpha=0.3,
        color='tomato', label='Feature count')

ax1.set_xlabel('Fourier Order', fontsize=12)
ax1.set_ylabel('Value Surface Variance', color='steelblue', fontsize=12)
ax2.set_ylabel('Number of Features', color='tomato', fontsize=12)
ax1.set_title('Fourier Basis Order vs. Approximation Complexity', fontsize=13)
ax1.set_xticks(orders)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_421():
    assert len(surface_variances) == len(orders), \
        f"Compute surface_variances for all orders. Got {len(surface_variances)}, expected {len(orders)}"
    assert all(v >= 0 for v in surface_variances), \
        "Variance should be non-negative"
    # Feature count should grow with order
    for i in range(len(orders)-1):
        assert n_features_list[i] < n_features_list[i+1], \
            f"Order {orders[i]} should have fewer features than order {orders[i+1]}"
    print("Exercise 4.2.1 passed!")
    print(f"Variance across orders: {[round(v,4) for v in surface_variances]}")

test_exercise_421()

### Exercise 4.2.2: Adaptive Learning Rate for Linear TD

**Task:** Sutton & Barto recommend setting alpha = `step_size / n_tilings` for tile coding to normalize for the number of active features. Implement an `AdaptiveAlpha` class that automatically scales the learning rate by `1 / n_active_features` where `n_active_features = phi.sum()`. Compare its convergence against fixed alpha on tile coding for 200 episodes.

<details>
<summary>Hint 1</summary>
In the weight update w += alpha * delta * phi, replace alpha with base_alpha / phi.sum(). For tile coding phi.sum() == n_tilings, so this gives the recommended alpha / n_tilings scaling automatically.
</details>

<details>
<summary>Hint 2</summary>
Write a modified version of semi_gradient_td0 that accepts a base_alpha and divides by phi.sum() at each step. Then compare episode length curves.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def semi_gradient_td0_adaptive(encoder, n_episodes, base_alpha, gamma=1.0, seed=42):
    """
    Semi-gradient TD(0) with adaptive learning rate:
    effective_alpha = base_alpha / phi.sum()

    This normalizes for the number of active features, ensuring updates
    have consistent magnitude regardless of feature vector sparsity.
    """
    env = gym.make('MountainCar-v0')
    w   = np.zeros(encoder.n_features)
    ep_lengths = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        done   = False
        steps  = 0

        while not done:
            phi_s  = encoder.encode(obs)
            action = env.action_space.sample()
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done   = terminated or truncated
            steps += 1

            V_s    = np.dot(w, phi_s)
            V_next = 0.0 if done else np.dot(w, encoder.encode(next_obs))
            delta  = reward + gamma * V_next - V_s

            # Adaptive alpha: scale by number of active features
            n_active  = max(phi_s.sum(), 1.0)   # avoid division by zero
            eff_alpha = base_alpha / n_active
            w        += eff_alpha * delta * phi_s

            obs = next_obs

        ep_lengths.append(steps)

    env.close()
    return w, ep_lengths


print("Training with fixed alpha...")
_, lengths_fixed = semi_gradient_td0(tc, n_episodes=200, alpha=0.1 / tc.n_tilings, seed=0)

print("Training with adaptive alpha...")
_, lengths_adaptive = semi_gradient_td0_adaptive(tc, n_episodes=200, base_alpha=0.1, seed=0)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(smooth(lengths_fixed),    color='steelblue', linewidth=2,
        label=f'Fixed alpha={0.1/tc.n_tilings:.4f}')
ax.plot(smooth(lengths_adaptive), color='tomato',    linewidth=2,
        label='Adaptive alpha (base=0.1 / n_active)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Length', fontsize=12)
ax.set_title('Fixed vs. Adaptive Alpha in Semi-Gradient TD(0)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_422():
    assert len(lengths_fixed) == 200 and len(lengths_adaptive) == 200, \
        "Both should run for 200 episodes"
    # Both should produce valid episode lengths (between 1 and 200 for Mountain Car)
    assert all(1 <= l <= 201 for l in lengths_fixed), \
        "Fixed: episode lengths should be in [1, 201]"
    assert all(1 <= l <= 201 for l in lengths_adaptive), \
        "Adaptive: episode lengths should be in [1, 201]"
    print("Exercise 4.2.2 passed!")
    print(f"Fixed alpha   — mean final length: {np.mean(lengths_fixed[-20:]):.0f}")
    print(f"Adaptive alpha — mean final length: {np.mean(lengths_adaptive[-20:]):.0f}")

test_exercise_422()

## Key Takeaways

1. **Semi-gradient TD(0)** extends tabular TD to continuous state spaces by replacing the lookup table with $\hat{V}(s; \mathbf{w}) = \mathbf{w}^\top \phi(s)$. The update $\mathbf{w} \leftarrow \mathbf{w} + \alpha \delta_t \phi(S_t)$ is simple and efficient.
2. **"Semi" gradient** means we don't differentiate through the bootstrap target. This is a deliberate choice — the full gradient would require storing extra terms and doesn't improve convergence in practice for prediction.
3. **Tile coding** is the most sample-efficient basis for RL: sparse features enable efficient updates, and overlapping tilings provide smooth generalization. The recommended alpha scaling is `step_size / n_tilings`.
4. **Polynomial and Fourier bases** are dense and better suited when the value function is known to be smooth. Fourier bases allow analysis by frequency components.
5. **The value surface** should show higher values near states from which the goal is reachable — on Mountain Car, high values appear near the right side and near high absolute velocity.

## What's Next

Module 5 replaces the linear function approximator with a **neural network** (nonlinear). Deep Q-Networks (DQN) inherit the same TD error structure but use backpropagation to update a nonlinear $\hat{Q}(s, a; \mathbf{\theta})$. Two additions — replay buffer and target network — are needed to stabilize training.